# Step 6: GPU check and AMALIA precision probe

One notebook, run top to bottom with **Run All** (about 10 min). It stops early, with a plain message,
if the hardware is wrong, so it cannot waste your quota.

Before running, in the right panel: **Accelerator = GPU T4 x2** and **Internet = on**.
No dataset is needed for this notebook.

Paste back to Claude the text printed by cell 2 and by cell 3.

In [ ]:
# 1. Setup (about 3 min): latest code from GitHub
%cd /kaggle/working
!pip -q install -U "transformers>=5.10" accelerate huggingface_hub
!rm -rf pt-legal-rag-eval && git clone -q -b step-6-runner https://github.com/cardohso/pt-legal-rag-eval.git
%cd pt-legal-rag-eval
!pip -q install -e .
!git rev-parse HEAD

In [ ]:
# 2. Hardware gate: stops here, before any download, unless this session has two T4 GPUs
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-m", "ptlegal_eval.gpucheck"], capture_output=True, text=True
)
print(check.stdout + check.stderr)
if check.returncode != 0:
    raise RuntimeError(
        "Stopped before any download. Fix what is printed above (wrong GPU, or the setup "
        "failed: check Internet = on), then Run All again."
    )

In [ ]:
# 3. Probe AMALIA in FP16 and BF16: where do values overflow, and how fast is each? (about 7 min)
!python -m ptlegal_eval.probe --models amalia --dtypes float16,bfloat16 --out /kaggle/working/probe

In [ ]:
# 4. One zip to download from the Output panel
!cd /kaggle/working && zip -rq probe.zip probe && ls -la probe.zip probe

Download `probe.zip` from the **Output** panel (right side, *Data*) if Claude asks for it.